In [24]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Passw0rd1!"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.getRecordCriteria({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True,errors='ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())),
    html.Center(html.B(html.H1('Joshua Trabka : CS 340 Dashboard'))),
    html.Hr(),
     dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All', 'value': 'All'},
            {'label':'Water Rescue', 'value': 'Water'},
            {'label':'Mountain Rescue', 'value': 'Mountain'},
            {'label':'Disaster Rescue', 'value':'Disaster'},
        ],
        value='All'
    ),
        


    
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
        filter_action="native", #allow a filter
        sort_action="native", #allow sorting
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native", #enable pagination
        page_current=0, #set start page
        page_size=10, #set rows per page


                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # If "Water Rescue" button is clicked, filter for 'Water'
    if filter_type == "Water":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
        df = pd.DataFrame.from_records(db.getRecordCriteria(query))
        
    # If "Mountain Rescue" button is clicked, filter for 'Mountain'
    elif filter_type == "Mountain":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepard", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
        df = pd.DataFrame.from_records(db.getRecordCriteria(query))
        
    # If "Disaster Rescue" button is clicked, filter for 'Disaster'
    elif filter_type == "Disaster":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepard", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
        df = pd.DataFrame.from_records(db.getRecordCriteria(query))
        
    # Default 'All' returns all documents
    else: # This covers the 'All' case
        df = pd.DataFrame.from_records(db.getRecordCriteria({}))  

    # MongoDB returns '_id' which can cause issues, so we drop it
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    
    return df.to_dict('records')
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('filter-type', 'value')]  #  Updates with filter buttons
)
def update_graphs(filter_type):
    # Queries for filters
    if filter_type == "Water":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
        df = pd.DataFrame.from_records(db.getRecordCriteria(query))
        
    elif filter_type == "Mountain":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepard", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
        df = pd.DataFrame.from_records(db.getRecordCriteria(query))
        
    elif filter_type == "Disaster":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepard", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
        df = pd.DataFrame.from_records(db.getRecordCriteria(query))
        
    else: # This covers the 'All' case
        df = pd.DataFrame.from_records(db.getRecordCriteria({}))

    # If the filtered dataframe is empty, don't generate a chart
    if df.empty:
        return html.H4("No data available for this filter.")

    # Create the pie chart from the full filtered dataframe
    figure = px.pie(df, names='breed', title=f'Animal Breeds for {filter_type} Rescue')
    
    # Prevent overflow
    figure.update_traces(textposition='inside', textinfo='percent+label')
    figure.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
    
    return dcc.Graph(figure=figure)


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # If there is no data in the table, the map will not be created
    if viewData is None or len(viewData) == 0:
        return [html.H4("No data to display on map.")]

    dff = pd.DataFrame.from_dict(viewData)
    
    # Set the row index. Default to 0 if no row is selected.
    row_index = 0
    if index is not None and len(index) > 0:
        row_index = index[0]

    
    
    
    lat = dff.iloc[row_index]['location_lat']
    lon = dff.iloc[row_index]['location_long']
    
    # Check if lat or lon are null
    if not pd.notna(lat) or not pd.notna(lon):
        return [html.H4(f"Location data not available for this animal.")]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            dl.Marker(position=[lat, lon],
                children=[
                dl.Tooltip(dff.iloc[row_index]['breed']), # Use column name
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row_index]['name']) # Use column name
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://equalmodular-exodusswing-3000.codio.io/proxy/8050/
